In [ ]:
!pip install pandas pyarrow openai

In [1]:
import openai

print(openai.__version__)

1.71.0


In [2]:
import pandas as pd

DATA_FILE = "data/spend_data.parquet"
df = pd.read_parquet(DATA_FILE)
df["delivery_date"] = pd.to_datetime(df["delivery_date"])
df["ordered_date"] = pd.to_datetime(df["ordered_date"])

print(df.tail(3))

                                 order_id department_name    vendor_name  \
997  c57e0e85-3a59-46c5-bdb7-faf657a9e7f0     Maintenance      ValueMart   
998  4dc7f752-7bda-456b-a467-effe80d510fe       Furniture     MegaVendor   
999  d2d435ab-1c36-4b66-a828-21262fc49a41     Electronics  FurniturePlus   

    item_name  item_quantity  unit_cost  total_cost        ordered_by  \
997     Tools              4     303.91     1215.64  Jeffrey Lawrence   
998   Cabinet              7     116.82      817.74  Maria Montgomery   
999    Tablet              5     432.82     2164.10  Anthony Gonzalez   

           ordered_date       approved_by       delivery_date  
997 2023-11-13 16:09:36     Lindsay Blair 2023-12-12 16:09:36  
998 2023-07-27 10:45:17      Susan Rogers 2023-08-22 10:45:17  
999 2023-04-29 10:35:18  Jeffrey Lawrence 2023-05-14 10:35:18  


In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import date
from openai import OpenAI
from opik.integrations.openai import track_openai


os.environ["OPIK_WORKSPACE"] = "noroozi"
os.environ["OPIK_PROJECT_NAME"] = "spend-assistant"

client = OpenAI()
client = track_openai(client)

class SpendAssistant:
    def __init__(self, model="gpt-4o", debug=False):
        self.debug = debug
        self.model = model

    def _log_debug(self, label, content):
        if self.debug:
            print(f"\n[DEBUG] {label}:\n{content}\n")

    def _chat(self, system_prompt, user_prompt):
        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )
        answer = response.choices[0].message.content.strip()
        return answer

    def get_pandas_expression(self, user_query, df_head):
        today = date.today().strftime("%B %d, %Y")
        system_prompt = (
            "You are working with a pandas dataframe in Python. "
            "The name of the dataframe is `df`.\n"
            f"This is the result of `print(df.head())`:\n{df_head}\n\n"
            "Follow these instructions:\n"
            "Convert the query to executable Python code using Pandas and Numpy. "
            "The code should represent a solution to the query."
            "The final line of code should be a Python expression that can be called with the `eval()` function.\n"
            f"If no day or month or year are mentioned, assume the current day, month and year respectively. Current date is {today}.\n"
            "If the query asks for a report based on time periods, such as daily, monthly, quarterly, annual, "
            "   generate a complete breakdown for each requested period.\n"
            "If the query involves identifying a maximum/minimum or calculating a value (e.g. total, sum, average), "
            "   return both the label and its numeric value.\n"
            "Generate a code that avoids execution errors or runtime errors such as division by zero\n"
            "Separate the statements with only semicolons.\n"
        )
        user_prompt = (
            "PRINT ONLY THE EXPRESSION.\n"
            "Do not quote the expression.\n\n"
            f"Query: {user_query}\n\n"
            "Expression:"
        )
        return self._chat(system_prompt, user_prompt)

    def generate_final_response(self, user_query, pandas_expression, query_results):
        system_prompt = (
            "You are a Spend Assistant providing meaningful, insightful, and concise responses based on spending data."
        )
        user_prompt = (
            f"Given a user question, synthesize a response from the query results.\n"
            "If results are 'Empty', acknowledge there are no results.\n"
            "If results are 'Error', respond with: 'I cannot answer this question.'\n"
            f"User question: {user_query}\n"
            f"Pandas expression: ```{pandas_expression}```\n"
            f"Question results: {query_results}\n\n"
            "Response:"
        )
        return self._chat(system_prompt, user_prompt)

    def execute_expression(self, expression, df):
        local_vars = {'pd': pd, 'np': np, 'df': df}
        try:
            statements = [s.strip() for s in expression.replace('\n', ';').split(';') if s.strip()]
            for stmt in statements[:-1]:
                exec(stmt, {}, local_vars)
            result = eval(statements[-1], {}, local_vars)

            # Empty results
            if result is None or (not isinstance(result, (pd.DataFrame, pd.Series, np.ndarray, list)) and pd.isna(result)):
                return "Empty"
            
            return result
        
        except ValueError as ve:
            self._log_debug("Execution ValueError", str(ve))
            return "Empty"
        
        except Exception as e:
            self._log_debug("Execution Error", str(e))
            return "Error"

    def search(self, user_query, df):
        self._log_debug("Question", user_query)

        expr = self.get_pandas_expression(user_query, df.head())

        self._log_debug("Execution Expression", expr)
        
        results = self.execute_expression(expr, df)
        
        self._log_debug("Search Results", results)
        
        return expr, results

    def answer(self, user_query, df):
        pandas_expr, results = self.search(user_query, df)
        final_response = self.generate_final_response(user_query, pandas_expr, results)

        self._log_debug("Final Response", final_response)
        
        return final_response

In [7]:
spend_assistant = SpendAssistant(model="gpt-4o", debug=True)

In [8]:
user_query = "Which department spent the most in 2023?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
Which department spent the most in 2023?


[DEBUG] Execution Expression:
df['ordered_date'] = pd.to_datetime(df['ordered_date']); df_2023 = df[df['ordered_date'].dt.year == 2023]; department_spending = df_2023.groupby('department_name')['total_cost'].sum(); max_spending_department = department_spending.idxmax(); max_spending_value = department_spending.max(); (max_spending_department, max_spending_value)


[DEBUG] Search Results:
('Maintenance', 123507.78)


[DEBUG] Final Response:
The department that spent the most in 2023 is the Maintenance department, with a total spending of $123,507.78.



In [7]:
user_query = "Who were the top 3 vendors for the department that spend the most?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
Who were the top 3 vendors for the department that spend the most?


[DEBUG] Execution Expression:
df['ordered_date'] = pd.to_datetime(df['ordered_date']); df['total_cost'] = pd.to_numeric(df['total_cost'], errors='coerce'); department_spending = df.groupby('department_name')['total_cost'].sum(); top_department = department_spending.idxmax(); top_vendors = df[df['department_name'] == top_department].groupby('vendor_name')['total_cost'].sum().nlargest(3); top_vendors


[DEBUG] Search Results:
vendor_name
OfficeMax Corp    43003.27
MegaVendor        36101.27
ElectroTech       35230.59
Name: total_cost, dtype: float64


[DEBUG] Final Response:
The top 3 vendors for the department with the highest spending are OfficeMax Corp with $43,003.27, MegaVendor with $36,101.27, and ElectroTech with $35,230.59.



In [10]:
user_query = "What is my total monthly spend?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
What is my total monthly spend?


[DEBUG] Execution Expression:
df['ordered_date'] = pd.to_datetime(df['ordered_date']); df['month_year'] = df['ordered_date'].dt.to_period('M'); monthly_spend = df.groupby('month_year')['total_cost'].sum(); monthly_spend


[DEBUG] Search Results:
month_year
2023-03    22726.32
2023-04    48214.83
2023-05    56042.78
2023-06    58842.88
2023-07    54117.38
2023-08    62244.16
2023-09    67863.41
2023-10    77359.80
2023-11    51144.98
2023-12    48259.30
2024-01    48231.75
2024-02    60406.52
2024-03    45859.41
2024-04    66457.39
2024-05    54869.04
2024-06    58272.80
2024-07    62506.55
2024-08    73798.95
2024-09    55271.95
2024-10    42967.47
2024-11    45513.67
2024-12    84386.69
2025-01    39351.93
2025-02    66616.26
2025-03    45109.78
Freq: M, Name: total_cost, dtype: float64


[DEBUG] Final Response:
Here is your total monthly spend from March 2023 to March 2025:

- March 2023: $22,726.32
- April 2023: $48,214.83
- May 2

In [9]:
user_query = "Who were the top 3 approvers?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
Who were the top 3 approvers?


[DEBUG] Execution Expression:
df['approved_by'].value_counts().head(3)


[DEBUG] Search Results:
approved_by
Lindsay Blair    68
Lisa Smith       57
Jill Rhodes      55
Name: count, dtype: int64


[DEBUG] Final Response:
The top 3 approvers are Lindsay Blair with 68 approvals, Lisa Smith with 57 approvals, and Jill Rhodes with 55 approvals.



In [10]:
user_query = "What are the top 3 departments by total spend?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
What are the top 3 departments by total spend?


[DEBUG] Execution Expression:
df.groupby('department_name')['total_cost'].sum().nlargest(3)


[DEBUG] Search Results:
department_name
Electronics        310595.70
Maintenance        289359.73
Office Supplies    271338.41
Name: total_cost, dtype: float64


[DEBUG] Final Response:
The top 3 departments by total spend are Electronics with $310,595.70, Maintenance with $289,359.73, and Office Supplies with $271,338.41.

